# SNAP Optimization Workflow


This notebook rewrites `examples/run_snap_optimization_demo.py` as a study-style tutorial. It uses the repo-side helper module under `examples/studies/snap_opt` to optimize a selective multitone pulse for a target set of per-manifold phases.

The physical abstraction is narrower than the full runtime simulator: the study helper works in a matched rotating frame with a simplified model of manifold-selective control. That makes it ideal for teaching how the optimization loop behaves, while staying honest that this is a repo-side study workflow rather than a canonical package entry point.


## Imports


In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

REPO_ROOT = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "pyproject.toml").exists() and (candidate / "cqed_sim").is_dir()
    ),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError("Could not resolve the repository root from the notebook working directory.")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import matplotlib.pyplot as plt
import numpy as np
import qutip as qt

from tutorials.workflow_tutorial_support import configure_notebook_style

configure_notebook_style()

from examples.studies.snap_opt import (
    SnapModelConfig,
    SnapRunConfig,
    SnapToneParameters,
    optimize_snap_parameters,
    target_difficulty_metric,
)


## Physics / model definition


In [ ]:
model = SnapModelConfig(n_cav=7, n_tr=2, chi=2.0 * np.pi * 0.02).build_model()
target_phases = np.array([0.0, 1.1, -0.7, 0.4], dtype=float)
run_config = SnapRunConfig(duration=170.0, dt=0.2, base_amp=0.010)
initial_params = SnapToneParameters.vanilla(target_phases)
difficulty = target_difficulty_metric(target_phases)

print("Target difficulty metric:", difficulty)


## Pulse / sequence construction


In [ ]:
print("Initial amplitudes:", initial_params.amplitudes)
print("Initial detunings:", initial_params.detunings)
print("Initial phases:", initial_params.phases)


## Simulation


In [ ]:
result = optimize_snap_parameters(
    model,
    target_phases,
    run_config,
    initial_params=initial_params,
    max_iter=40,
    learning_rate=0.3,
    threshold=6e-3,
)

iterations = np.arange(len(result.history_error), dtype=int)
print("Converged:", result.converged)
print("Initial mean overlap error:", float(result.history_error[0]))
print("Final mean overlap error:", float(result.history_error[-1]))


## Analysis / visualization


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11.0, 4.0))

axes[0].plot(iterations, result.history_error, "o-", color="#4C78A8")
axes[0].set_xlabel("Optimization iteration")
axes[0].set_ylabel("Mean overlap error")
axes[0].set_title("SNAP optimization progress")

tone_index = np.arange(result.params.amplitudes.size, dtype=int)
axes[1].bar(tone_index - 0.25, result.params.amplitudes, width=0.25, label="amplitude")
axes[1].bar(tone_index, result.params.detunings, width=0.25, label="detuning")
axes[1].bar(tone_index + 0.25, result.params.phases, width=0.25, label="phase")
axes[1].set_xlabel("Tone / manifold index")
axes[1].set_title("Optimized tone parameters")
axes[1].legend(loc="best")

plt.tight_layout()
plt.show()


## Interpretation


The optimization history shows how quickly the selective slow-stage model can reduce coherent manifold errors for this target. The final parameter bars are not generic hardware calibrations; they are the result of this specific study model and target.

This notebook is best thought of as a bridge between the reusable package and a more specialized control study. It teaches the workflow honestly without pretending that every helper in `examples/studies` is part of the stable public API.


## Variations / exercises


- Change `target_phases` to see how target difficulty affects convergence.
- Increase `max_iter` and compare the cost of extra optimization budget.
- Follow this notebook with the paper-reproduction notebooks under `test_against_papers/` if you want literature-oriented validation rather than a workflow demo.
